In [1]:
import os
import pathlib
import sys
import time

import pandas as pd
import psutil
from image_analysis_3D.featurization_utils.feature_writing_utils import (
    format_morphology_feature_name,
)
from image_analysis_3D.featurization_utils.intensity_utils import (
    measure_3D_intensity_CPU,
)
from image_analysis_3D.featurization_utils.loading_classes import (
    ImageSetLoader,
    ObjectLoader,
)
from image_analysis_3D.featurization_utils.resource_profiling_util import (
    get_mem_and_time_profiling,
)
from image_analysis_3D.file_utils.arg_parsing_utils import (
    check_for_missing_args,
    parse_args,
)
from image_analysis_3D.file_utils.notebook_init_utils import (
    bandicoot_check,
    init_notebook,
)

root_dir, in_notebook = init_notebook()
image_base_dir = bandicoot_check(
    pathlib.Path(os.path.expanduser("~/mnt/bandicoot")).resolve(), root_dir
)

In [2]:
if not in_notebook:
    arguments_dict = parse_args()
    patient = arguments_dict["patient"]
    well_fov = arguments_dict["well_fov"]
    channel = arguments_dict["channel"]
    compartment = arguments_dict["compartment"]
    processor_type = arguments_dict["processor_type"]
    input_subparent_name = arguments_dict["input_subparent_name"]
    mask_subparent_name = arguments_dict["mask_subparent_name"]
    output_features_subparent_name = arguments_dict["output_features_subparent_name"]

else:
    well_fov = "D2-3"
    patient = "NF0014_T1"
    channel = "Mito"
    compartment = "Cytoplasm"
    processor_type = "CPU"
    input_subparent_name = "zstack_images"
    mask_subparent_name = "segmentation_masks"
    output_features_subparent_name = "extracted_features"

image_set_path = pathlib.Path(
    f"{image_base_dir}/data/{patient}/{input_subparent_name}/{well_fov}/"
)
mask_set_path = pathlib.Path(
    f"{image_base_dir}/data/{patient}/{mask_subparent_name}/{well_fov}/"
)
output_parent_path = pathlib.Path(
    f"{image_base_dir}/data/{patient}/{output_features_subparent_name}/{well_fov}/"
)
output_parent_path.mkdir(parents=True, exist_ok=True)

In [3]:
channel_n_compartment_mapping = {
    "DNA": "405",
    "AGP": "488",
    "ER": "555",
    "Mito": "640",
    "BF": "TRANS",
    "Nuclei": "nuclei_",
    "Cell": "cell_",
    "Cytoplasm": "cytoplasm_",
    "Organoid": "organoid_",
}

In [4]:
start_time = time.time()
# get starting memory (cpu)
start_mem = psutil.Process(os.getpid()).memory_info().rss / 1024**2

In [5]:
image_set_loader = ImageSetLoader(
    image_set_path=image_set_path,
    mask_set_path=mask_set_path,
    anisotropy_spacing=(1, 0.1, 0.1),
    channel_mapping=channel_n_compartment_mapping,
    image_set_name=well_fov,
)

In [6]:
object_loader = ObjectLoader(
    image_set_loader.image_set_dict[channel],
    image_set_loader.image_set_dict[compartment],
    channel,
    compartment,
)
print(object_loader.image.shape, object_loader.label_image.shape)

(17, 1537, 1540) (17, 1537, 1540)


In [7]:
if processor_type == "CPU":
    output_dict = measure_3D_intensity_CPU(object_loader)
else:
    raise ValueError(f"Processor type {processor_type} is not supported. Use 'CPU'.")
final_df = pd.DataFrame(output_dict)
# prepend compartment and channel to column names
final_df = final_df.pivot(
    index=["object_id"],
    columns="feature_name",
    values="value",
).reset_index()
final_df.rename(
    columns={
        col: format_morphology_feature_name(
            compartment=compartment,
            channel=channel,
            feature_type="Granularity",
            measurement=col,
        )
        if col != "object_id"
        else col
        for col in final_df.columns
    },
    inplace=True,
)

final_df.insert(0, "image_set", image_set_loader.image_set_name)

output_file = pathlib.Path(
    output_parent_path
    / f"Intensity_{compartment}_{channel}_{processor_type}_features.parquet"
)
output_file.parent.mkdir(parents=True, exist_ok=True)
final_df.to_parquet(output_file)
final_df.head()

feature_name,image_set,object_id,Cytoplasm_Mito_Granularity_CMI-X,Cytoplasm_Mito_Granularity_CMI-Y,Cytoplasm_Mito_Granularity_CMI-Z,Cytoplasm_Mito_Granularity_IntegratedIntensity,Cytoplasm_Mito_Granularity_IntegratedIntensityEdge,Cytoplasm_Mito_Granularity_LowerQuartileIntensity,Cytoplasm_Mito_Granularity_MassDisplacement,Cytoplasm_Mito_Granularity_MaxIntensity,...,Cytoplasm_Mito_Granularity_MaxZ,Cytoplasm_Mito_Granularity_MeanAbsoluteDeviationIntensity,Cytoplasm_Mito_Granularity_MeanIntensity,Cytoplasm_Mito_Granularity_MeanIntensityEdge,Cytoplasm_Mito_Granularity_MedianIntensity,Cytoplasm_Mito_Granularity_MinIntensity,Cytoplasm_Mito_Granularity_MinIntensityEdge,Cytoplasm_Mito_Granularity_StdIntensity,Cytoplasm_Mito_Granularity_StdIntensityEdge,Cytoplasm_Mito_Granularity_UpperQuartileIntensity
0,D2-3,257,399.981873,1363.669312,4.574067,458468992.0,36538460.0,3341.0,2.182631,25443.0,...,1.0,1089.915894,4299.463379,1981.478394,3855.0,1799.0,0.0,1552.755737,2294.058350,4883.0
1,D2-3,771,997.446411,527.880310,2.494421,7646778.0,2385217.0,3598.0,1.183681,7967.0,...,2.0,726.573975,4276.721680,1896.038940,4112.0,2313.0,0.0,880.080750,2162.920410,4883.0
2,D2-3,1028,1018.719543,618.999146,2.553084,24018192.0,6248698.0,3341.0,1.139691,7453.0,...,1.0,981.493286,4334.631348,1871.427979,4369.0,2056.0,0.0,1111.643188,2100.825928,5140.0
3,D2-3,1285,940.743530,685.141602,6.155763,990793856.0,50458096.0,2827.0,2.037082,9509.0,...,0.0,783.930481,3555.374023,1722.766113,3084.0,1542.0,0.0,1013.282593,1853.331787,3855.0
4,D2-3,1799,928.888916,535.070312,6.477686,272892864.0,24250776.0,3084.0,1.123300,7967.0,...,2.0,749.689758,3712.222168,1730.344360,3341.0,1542.0,0.0,927.256287,1830.787231,4369.0


In [8]:
end_mem = psutil.Process(os.getpid()).memory_info().rss / 1024**2
end_time = time.time()
get_mem_and_time_profiling(
    start_mem=start_mem,
    end_mem=end_mem,
    start_time=start_time,
    end_time=end_time,
    feature_type="Intensity",
    well_fov=well_fov,
    patient_id=patient,
    channel=channel,
    compartment=compartment,
    CPU_GPU=processor_type,
    output_file_dir=pathlib.Path(
        f"{root_dir}/data/{patient}/extracted_features/run_stats/{well_fov}_{channel}_{compartment}_Intensity_{processor_type}.parquet"
    ),
)


        Memory and time profiling for the run:
        Patient ID: NF0014_T1
        Well and FOV: D2-3
        Feature type: Intensity
        CPU/GPU: CPU
        Memory usage: 988.31 MB
        Time elapsed:
        --- 21.84 seconds ---
        --- 0.36 minutes ---
        --- 0.01 hours ---
    


True